# 02 - Business topics first, then the topics nobody defined yet

**Business question:** which topics do customers raise on calls, including new ones the team has not named yet?

| Section | What it shows |
|---|---|
| 1 | Seeded assignment: each segment matched to the business-defined topics with SQL vector similarity |
| 2 | Discovery on the leftovers: UMAP + HDBSCAN in a warehouse stored procedure; noise stays unassigned |
| 3 | Why density clustering and not k-means: k-means forces every leftover into a topic |
| 4 | The fitted clusterer is a versioned model in the Model Registry, scored from SQL |
| 5 | An LLM names each new cluster from a few representatives; names wait for review |
| 6 | January arrives: run 2 (V2), matched to run 1 by member overlap |
| 7 | `ML.SEGMENT_TOPICS`: one topic per segment for everything downstream |

Runs standalone after `01`. Every section ends in a `CHECK`.

In [ ]:
NOTEBOOK_TAG = "transcript-churn-02"

# Session: Snowsight Workspaces supplies one; a local kernel (Cortex Code Desktop,
# ./demo.sh build) opens the connection saved in .env by ./demo.sh setup.
import json
import os
import time

import numpy as np
import pandas as pd

DB = "TRANSCRIPT_CHURN_ML_DEMO"
WH = "TRANSCRIPT_CHURN_ML_WH"
ROLE = "SYSADMIN"
EMBED_MODEL = "snowflake-arctic-embed-l-v2.0"
LLM = "claude-sonnet-4-5"
# The account this demo is allowed to build in. ./demo.sh setup writes it to .env;
# in Workspaces, replace the placeholder with your demo account name.
DEMO_ACCOUNT = os.environ.get("DEMO_EXPECTED_ACCOUNT", "<YOUR_DEMO_ACCOUNT>")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    LOCAL = False
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config(
        "connection_name", os.environ.get("SNOWFLAKE_DEFAULT_CONNECTION_NAME", "default")).create()
    LOCAL = True

session.use_role(ROLE)
session.use_warehouse(WH)
session.use_database(DB)
session.sql(f"ALTER SESSION SET QUERY_TAG = '{NOTEBOOK_TAG}'").collect()

if LOCAL:
    # Local kernels have no SQL cells: run each %%sql cell through this session instead.
    from IPython.core.magic import register_cell_magic
    from IPython.display import display

    @register_cell_magic
    def sql(line, cell):
        args = line.split()
        name = args[args.index("-r") + 1] if "-r" in args else None
        rows = session.sql(cell).collect()      # collect(): CALL and EXECUTE cannot be described first
        result = pd.DataFrame([r.as_dict() for r in rows])
        if name:
            get_ipython().user_ns[name] = result
        display(result.head(50))

CHECKS = {}


def check(section, ok, evidence):
    status = "PASS" if ok else "FAIL"
    CHECKS[section] = (status, evidence)
    print(f"CHECK {section}: {status} | {evidence}")
    if not ok:
        raise AssertionError(f"CHECK {section} failed: fix the cause and re-run this section")


def q(text):
    return session.sql(text).to_pandas()


def one(text):
    return session.sql(text).collect()[0][0]


account, role = session.sql("SELECT CURRENT_ACCOUNT_NAME(), CURRENT_ROLE()").collect()[0]
if account != DEMO_ACCOUNT:
    raise SystemExit(f"STOP: connected to {account}, but this demo may only build in {DEMO_ACCOUNT}. "
                     "Set DEMO_EXPECTED_ACCOUNT (./demo.sh setup does) or edit DEMO_ACCOUNT in this cell.")
if not session.sql(f"SHOW SCHEMAS LIKE 'RAW' IN DATABASE {DB}").collect():
    raise SystemExit(f"STOP: {DB}.RAW is missing. Run lab/setup.sql (./demo.sh setup) first.")
print(f"Connected as role {role} on {account}, warehouse {WH}, query tag {NOTEBOOK_TAG} "
      f"({'local kernel' if LOCAL else 'Workspaces session'})")

In [ ]:
n_emb = one(f"SELECT COUNT(*) FROM {DB}.CORE.SEGMENT_EMBEDDINGS")
if n_emb == 0:
    raise SystemExit("STOP: CORE.SEGMENT_EMBEDDINGS is empty. Run notebook 01 first.")
HISTORY_CUTOFF = "2026-01-01"
# Seeded rule, tuned on this data: best similarity to a topic's exemplars, and its margin over the runner-up
SIM_MIN, MARGIN_MIN = 0.47, 0.03
session.sql(f"DROP MODEL IF EXISTS {DB}.ML.TOPIC_CLUSTERER").collect()     # re-runnable: versions V1/V2 are rebuilt
print(f"{n_emb:,} embedded segments; seeded rule: similarity >= {SIM_MIN} and margin >= {MARGIN_MIN}")

---
## 1. Seeded assignment against the business taxonomy

The team defined eight topics in `ML.TOPIC_TAXONOMY`, two short exemplars each, written in their own words. Exemplars are
embedded with the same model as the segments. A segment takes a topic when its best cosine similarity clears a floor
**and** beats the runner-up by a margin; anything ambiguous is left for discovery. Against hundreds of exemplars and a few
thousand segments a month this is a small join, so it stays in SQL.

In [ ]:
%%sql -r embed_exemplars
CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_EXEMPLAR_EMBEDDINGS
  COMMENT = 'Business exemplars embedded with the same model as the segments'
AS SELECT TOPIC_ID, TOPIC_NAME, EXEMPLAR_TEXT, 'snowflake-arctic-embed-l-v2.0' AS EMBED_MODEL,
          AI_EMBED('snowflake-arctic-embed-l-v2.0', EXEMPLAR_TEXT) AS EMBEDDING
FROM TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_TAXONOMY

In [ ]:
t0 = time.time()
session.sql(f"""
CREATE OR REPLACE TABLE {DB}.ML.SEEDED_ASSIGNMENTS COMMENT = 'Best business topic per segment; ASSIGNED when it clears the rule' AS
WITH sims AS (
  SELECT s.SEGMENT_ID, e.TOPIC_ID, MAX(VECTOR_COSINE_SIMILARITY(s.EMBEDDING, e.EMBEDDING)) AS SIM
  FROM {DB}.CORE.SEGMENT_EMBEDDINGS s CROSS JOIN {DB}.ML.TOPIC_EXEMPLAR_EMBEDDINGS e GROUP BY 1, 2),
ranked AS (
  SELECT SEGMENT_ID, TOPIC_ID, SIM,
         ROW_NUMBER() OVER (PARTITION BY SEGMENT_ID ORDER BY SIM DESC) AS RN,
         LEAD(SIM) OVER (PARTITION BY SEGMENT_ID ORDER BY SIM DESC) AS SIM2
  FROM sims)
SELECT SEGMENT_ID, TOPIC_ID AS BEST_TOPIC, SIM AS BEST_SIM, SIM - SIM2 AS MARGIN,
       SIM >= {SIM_MIN} AND SIM - SIM2 >= {MARGIN_MIN} AS ASSIGNED
FROM ranked WHERE RN = 1""").collect()
SEED_SECONDS = time.time() - t0
pairs = one(f"SELECT COUNT(*) * (SELECT COUNT(*) FROM {DB}.ML.TOPIC_EXEMPLAR_EMBEDDINGS) FROM {DB}.CORE.SEGMENT_EMBEDDINGS")
print(f"{pairs:,} segment-exemplar similarities in {SEED_SECONDS:.1f} s")

In [ ]:
%%sql -r seeded_by_topic
SELECT IFF(ASSIGNED, BEST_TOPIC, '(left for discovery)') AS TOPIC, COUNT(*) AS SEGMENTS, ROUND(AVG(BEST_SIM), 3) AS AVG_SIM
FROM TRANSCRIPT_CHURN_ML_DEMO.ML.SEEDED_ASSIGNMENTS GROUP BY 1 ORDER BY 2 DESC

In [ ]:
# RAW.SEED_TRUTH is the generator's answer key: used here only to grade, never as an input
g = q(f"""
WITH truth AS (SELECT DISTINCT CALL_ID || '-' || FLOOR((TURN_NO - 1) / 4) AS SEGMENT_ID, TRUE_TOPIC FROM {DB}.RAW.SEED_TRUTH)
SELECT AVG(IFF(a.ASSIGNED, 1, 0)) AS SEEDED_SHARE,
       AVG(IFF(a.ASSIGNED, IFF(a.BEST_TOPIC = t.TRUE_TOPIC, 1, 0), NULL)) AS SEEDED_ACCURACY,
       COUNT_IF(NOT a.ASSIGNED) AS LEFTOVERS
FROM {DB}.ML.SEEDED_ASSIGNMENTS a JOIN truth t USING (SEGMENT_ID)""").iloc[0]
SEEDED_SHARE, SEEDED_ACC = float(g.SEEDED_SHARE), float(g.SEEDED_ACCURACY)
check("S1", SEEDED_SHARE > 0.5 and SEEDED_ACC >= 0.95,
      f"seeded topics claim {SEEDED_SHARE:.1%} of segments at {SEEDED_ACC:.1%} accuracy (graded against the answer key); "
      f"{int(g.LEFTOVERS):,} leftovers go to discovery; similarity join {SEED_SECONDS:.1f} s")

---
## 2. Discover topics in the leftovers

A Python stored procedure on the demo warehouse reads the leftover embeddings, reduces them with **UMAP** (1,024 to 5
dimensions) and clusters them with **HDBSCAN**. HDBSCAN finds dense groups and labels everything else `-1` (noise) instead of
forcing it into a topic. umap-learn, hdbscan and scikit-learn come from the Snowflake package channel, so nothing is
installed by hand and the rows never leave Snowflake.

The procedure also logs the fitted reducer + clusterer as a versioned model (Section 4) and writes every member to
`ML.CLUSTER_MEMBERS`.

In [ ]:
%%sql -r create_cluster_support
EXECUTE IMMEDIATE $$
BEGIN
  CREATE STAGE IF NOT EXISTS TRANSCRIPT_CHURN_ML_DEMO.ML.CODE_STAGE COMMENT = 'Stored procedure code';
  CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.ML.CLUSTER_MEMBERS (
    RUN_LABEL STRING, SEGMENT_ID STRING, CLUSTER_ID INT, STRENGTH FLOAT, KMEANS_ID INT)
    COMMENT = 'Every leftover segment per discovery run: HDBSCAN cluster (-1 = noise) and the k-means comparison';
  CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.ML.CLUSTER_RUNS (
    RUN_LABEL STRING, RAN_AT TIMESTAMP_NTZ, TRAINING_SNAPSHOT STRING, STATS VARIANT)
    COMMENT = 'One row per discovery run and the training snapshot behind its model version';
  RETURN 'ML.CODE_STAGE, ML.CLUSTER_MEMBERS, ML.CLUSTER_RUNS ready';
END;
$$

In [ ]:
from snowflake.snowpark import Session

CLUSTER_PACKAGES = ["snowflake-snowpark-python", "snowflake-ml-python==2.4.0", "umap-learn==0.5.11", "hdbscan==0.8.33",
                    "scikit-learn==1.7.2", "joblib", "numpy", "pandas"]


def discover_topics(session: Session, run_label: str, cutoff_ts: str) -> str:
    import json as _j
    import joblib
    import numpy as _np
    import pandas as _pd
    import hdbscan
    import umap
    from sklearn.cluster import KMeans
    from snowflake.ml.model import custom_model, model_signature
    from snowflake.ml.registry import Registry

    db = "TRANSCRIPT_CHURN_ML_DEMO"
    snap = f"{db}.ML.CLUSTER_TRAINING_{run_label}"
    # An immutable snapshot of exactly what this version was fitted on
    session.sql(f"""CREATE OR REPLACE TABLE {snap} COMMENT = 'Leftover segments the {run_label} clusterer was fitted on' AS
                    SELECT e.SEGMENT_ID, e.EMBEDDING::ARRAY AS EMB FROM {db}.CORE.SEGMENT_EMBEDDINGS e
                    JOIN {db}.ML.SEEDED_ASSIGNMENTS a USING (SEGMENT_ID)
                    WHERE NOT a.ASSIGNED AND e.CALL_TS < '{cutoff_ts}'::TIMESTAMP_NTZ""").collect()
    df = session.table(snap).sort("SEGMENT_ID").to_pandas()
    X = _np.vstack([_np.asarray(_j.loads(v) if isinstance(v, str) else v, dtype=float) for v in df["EMB"]])
    red = umap.UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=42).fit(X)
    # A topic needs about a tenth of the leftovers (at least 25 segments) to count; smaller groups stay noise
    mcs = max(25, len(df) // 10)
    hdb = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10, prediction_data=True).fit(red.embedding_)
    labels, strength = hdb.labels_, hdb.probabilities_
    k = max(2, len(set(labels)) - (1 if -1 in labels else 0))
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    members = _pd.DataFrame({"RUN_LABEL": run_label, "SEGMENT_ID": df["SEGMENT_ID"], "CLUSTER_ID": labels.astype(int),
                             "STRENGTH": strength.astype(float), "KMEANS_ID": km.labels_.astype(int)})
    session.sql(f"DELETE FROM {db}.ML.CLUSTER_MEMBERS WHERE RUN_LABEL = '{run_label}'").collect()
    session.write_pandas(members, "CLUSTER_MEMBERS", database=db, schema="ML", auto_create_table=False)

    joblib.dump(red, "/tmp/umap.joblib")
    joblib.dump(hdb, "/tmp/hdbscan.joblib")

    class TopicClusterer(custom_model.CustomModel):
        """UMAP reducer + HDBSCAN clusterer; assigns new segments to a fitted cluster or -1 (noise)."""

        def __init__(self, context):
            super().__init__(context)
            import joblib as jl
            self.red = jl.load(self.context.path("umap"))
            self.hdb = jl.load(self.context.path("hdbscan"))

        @custom_model.inference_api
        def predict(self, input_df: _pd.DataFrame) -> _pd.DataFrame:
            import hdbscan as h
            import numpy as n
            emb = n.vstack([n.asarray(v, dtype=float) for v in input_df["EMB"]])
            lab, prob = h.approximate_predict(self.hdb, self.red.transform(emb))
            return _pd.DataFrame({"CLUSTER_ID": lab.astype(int), "STRENGTH": prob.astype(float)})

    sig = model_signature.ModelSignature(
        inputs=[model_signature.FeatureSpec(name="EMB", dtype=model_signature.DataType.DOUBLE, shape=(1024,))],
        outputs=[model_signature.FeatureSpec(name="CLUSTER_ID", dtype=model_signature.DataType.INT64),
                 model_signature.FeatureSpec(name="STRENGTH", dtype=model_signature.DataType.DOUBLE)])
    reg = Registry(session=session, database_name=db, schema_name="ML")
    mv = reg.log_model(
        TopicClusterer(custom_model.ModelContext(artifacts={"umap": "/tmp/umap.joblib", "hdbscan": "/tmp/hdbscan.joblib"})),
        model_name="TOPIC_CLUSTERER", version_name=run_label, signatures={"predict": sig},
        conda_dependencies=["umap-learn==0.5.11", "hdbscan==0.8.33", "scikit-learn==1.7.2", "joblib"],
        target_platforms=["WAREHOUSE"],
        comment=f"UMAP(5d, cosine) + HDBSCAN(min_cluster_size={mcs}) fitted on {snap} ({len(df)} leftover segments)")
    stats = {"run": run_label, "segments": int(len(df)), "clusters": int(len(set(labels)) - (1 if -1 in labels else 0)),
             "noise": int((labels == -1).sum()), "kmeans_k": int(k), "min_cluster_size": int(mcs), "training_snapshot": snap}
    for key in ("segments", "clusters", "noise"):
        mv.set_metric(key, stats[key])
    mv.set_metric("training_snapshot", snap)
    session.sql(f"DELETE FROM {db}.ML.CLUSTER_RUNS WHERE RUN_LABEL = '{run_label}'").collect()
    session.sql(f"INSERT INTO {db}.ML.CLUSTER_RUNS SELECT ?, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, ?, PARSE_JSON(?)",
                params=[run_label, snap, _j.dumps(stats)]).collect()
    return _j.dumps(stats)


session.sproc.register(discover_topics, name=f"{DB}.ML.DISCOVER_TOPICS", is_permanent=True, replace=True,
                       stage_location=f"@{DB}.ML.CODE_STAGE", packages=CLUSTER_PACKAGES, execute_as="caller",
                       comment="UMAP + HDBSCAN on leftover segments; logs the fitted clusterer to the Model Registry")
print(f"Registered {DB}.ML.DISCOVER_TOPICS(VARCHAR, VARCHAR) with {CLUSTER_PACKAGES}")

In [ ]:
%%sql -r discover_run_v1
CALL TRANSCRIPT_CHURN_ML_DEMO.ML.DISCOVER_TOPICS('V1', '2026-01-01')

In [ ]:
V1 = json.loads(discover_run_v1.iloc[0, 0])
truth_fit = q(f"""
WITH truth AS (SELECT DISTINCT CALL_ID || '-' || FLOOR((TURN_NO - 1) / 4) AS SEGMENT_ID, TRUE_TOPIC FROM {DB}.RAW.SEED_TRUTH),
m AS (SELECT c.*, t.TRUE_TOPIC FROM {DB}.ML.CLUSTER_MEMBERS c JOIN truth t USING (SEGMENT_ID) WHERE c.RUN_LABEL = 'V1'),
pairs AS (
  SELECT h.TRUE_TOPIC, c.CLUSTER_ID,
         COUNT_IF(m2.TRUE_TOPIC = h.TRUE_TOPIC AND m2.CLUSTER_ID = c.CLUSTER_ID) AS BOTH_N,
         COUNT_IF(m2.TRUE_TOPIC = h.TRUE_TOPIC OR m2.CLUSTER_ID = c.CLUSTER_ID) AS EITHER_N
  FROM (SELECT DISTINCT TRUE_TOPIC FROM m WHERE TRUE_TOPIC IN ('AI_LISTING_ASSISTANT', 'TEAM_SEAT_SHARING')) h
  CROSS JOIN (SELECT DISTINCT CLUSTER_ID FROM m WHERE CLUSTER_ID >= 0) c CROSS JOIN m m2
  GROUP BY 1, 2)
SELECT TRUE_TOPIC, MAX(BOTH_N / EITHER_N) AS BEST_JACCARD FROM pairs GROUP BY 1 ORDER BY 1""")
HIDDEN_MATCH = dict(zip(truth_fit.TRUE_TOPIC, truth_fit.BEST_JACCARD.astype(float)))
check("S2", len(HIDDEN_MATCH) == 2 and min(HIDDEN_MATCH.values()) >= 0.6 and V1["noise"] > 0,
      f"V1 on {V1['segments']} leftovers: {V1['clusters']} clusters, {V1['noise']} noise; hidden-topic best match "
      + ", ".join(f"{k} {v:.2f}" for k, v in HIDDEN_MATCH.items()) + " (Jaccard vs answer key, >= 0.6)")

---
## 3. Why not k-means?

k-means needs the number of topics up front (here it is handed the number HDBSCAN found, which you would not know in
practice) and puts **every** segment in a cluster, including the stray remarks that belong to no clear topic. HDBSCAN
leaves those as noise. On clean synthetic text both are about equally pure; the difference is the forced assignment, which
on real calls is where small talk and cross-talk pollute a new topic's member list.

In [ ]:
%%sql -r kmeans_vs_hdbscan
WITH truth AS (SELECT DISTINCT CALL_ID || '-' || FLOOR((TURN_NO - 1) / 4) AS SEGMENT_ID, TRUE_TOPIC FROM TRANSCRIPT_CHURN_ML_DEMO.RAW.SEED_TRUTH),
m AS (SELECT c.*, t.TRUE_TOPIC FROM TRANSCRIPT_CHURN_ML_DEMO.ML.CLUSTER_MEMBERS c JOIN truth t USING (SEGMENT_ID) WHERE RUN_LABEL = 'V1'),
h AS (SELECT CLUSTER_ID AS ID, TRUE_TOPIC, COUNT(*) AS N FROM m WHERE CLUSTER_ID >= 0 GROUP BY 1, 2),
k AS (SELECT KMEANS_ID AS ID, TRUE_TOPIC, COUNT(*) AS N FROM m GROUP BY 1, 2),
purity AS (
  SELECT 'HDBSCAN' AS METHOD, SUM(MAXN) / SUM(TOT) AS PURITY FROM (SELECT ID, MAX(N) AS MAXN, SUM(N) AS TOT FROM h GROUP BY 1)
  UNION ALL
  SELECT 'k-means', SUM(MAXN) / SUM(TOT) FROM (SELECT ID, MAX(N) AS MAXN, SUM(N) AS TOT FROM k GROUP BY 1))
SELECT p.METHOD,
       ROUND(IFF(p.METHOD = 'HDBSCAN', (SELECT COUNT_IF(CLUSTER_ID >= 0) / COUNT(*) FROM m), 1) * 100, 1) AS PCT_ASSIGNED,
       ROUND(p.PURITY * 100, 1) AS PCT_PURE_MEMBERS
FROM purity p ORDER BY 1

In [ ]:
kv = kmeans_vs_hdbscan.set_index("METHOD")
HDB_ASSIGNED, KM_ASSIGNED = float(kv.loc["HDBSCAN", "PCT_ASSIGNED"]), float(kv.loc["k-means", "PCT_ASSIGNED"])
HDB_PURITY, KM_PURITY = float(kv.loc["HDBSCAN", "PCT_PURE_MEMBERS"]), float(kv.loc["k-means", "PCT_PURE_MEMBERS"])
ABSORBED = one(f"SELECT COUNT(*) FROM {DB}.ML.CLUSTER_MEMBERS WHERE RUN_LABEL = 'V1' AND CLUSTER_ID = -1")
check("S3", KM_ASSIGNED == 100.0 and HDB_ASSIGNED < 100.0 and ABSORBED > 0,
      f"k-means (k={V1['kmeans_k']}, given) assigns {KM_ASSIGNED:.0f}% at {KM_PURITY:.1f}% purity, absorbing the "
      f"{ABSORBED} remarks HDBSCAN left as noise; HDBSCAN assigns {HDB_ASSIGNED:.1f}% at {HDB_PURITY:.1f}% purity")

---
## 4. The clusterer is a model version

`ML.TOPIC_CLUSTERER` holds the fitted UMAP + HDBSCAN pair as a custom model with one input, the 1,024-value embedding
array (a model method takes at most 500 arguments, so a vector goes in as a single array column). Its comment and metrics
name the training snapshot. It scores new segments from plain SQL with `MODEL(...)!PREDICT`, so the same assignment can
run in a dynamic table, a task or dbt.

In [ ]:
%%sql -r show_clusterer_versions
SHOW VERSIONS IN MODEL TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_CLUSTERER

In [ ]:
%%sql -r score_from_sql
SELECT e.SEGMENT_ID,
       TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_CLUSTERER!PREDICT(e.EMBEDDING::ARRAY):CLUSTER_ID::INT AS CLUSTER_ID,
       ROUND(TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_CLUSTERER!PREDICT(e.EMBEDDING::ARRAY):STRENGTH::FLOAT, 3) AS STRENGTH
FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENT_EMBEDDINGS e
JOIN TRANSCRIPT_CHURN_ML_DEMO.ML.SEEDED_ASSIGNMENTS a USING (SEGMENT_ID)
WHERE NOT a.ASSIGNED AND e.CALL_TS >= '2026-01-01' ORDER BY e.SEGMENT_ID LIMIT 10

In [ ]:
versions = [r.as_dict() for r in session.sql(f"SHOW VERSIONS IN MODEL {DB}.ML.TOPIC_CLUSTERER").collect()]
v1 = next(v for v in versions if v["name"] == "V1")
meta = json.loads(v1["metadata"]) if v1.get("metadata") else {}
check("S4", len(score_from_sql) == 10 and "CLUSTER_TRAINING_V1" in json.dumps(meta),
      f"versions {[v['name'] for v in versions]}; V1 metrics {meta.get('metrics')}; "
      f"10 January leftovers scored from SQL, clusters {sorted(score_from_sql.CLUSTER_ID.unique().tolist())}")

---
## 5. Name new clusters, then review

An LLM sees only a few of each cluster's most typical (redacted) remarks and the existing topic list, and returns a label,
a one-sentence description and whether the cluster duplicates an existing topic, as structured JSON. Each proposal lands
in `ML.PROPOSED_TOPICS` with status `PENDING` until someone approves it. One `AI_COMPLETE` call per cluster, not per
segment.

In [ ]:
%%sql -r create_proposed_topics
CREATE TABLE IF NOT EXISTS TRANSCRIPT_CHURN_ML_DEMO.ML.PROPOSED_TOPICS (
  RUN_LABEL STRING, CLUSTER_ID INT, MEMBERS INT, LABEL STRING, DESCRIPTION STRING, DUPLICATES STRING,
  STATUS STRING, PROPOSED_AT TIMESTAMP_NTZ, REVIEWED_BY STRING)
  COMMENT = 'LLM-proposed names for discovered clusters, waiting for review'

In [ ]:
def name_clusters(run_label):
    session.sql(f"DELETE FROM {DB}.ML.PROPOSED_TOPICS WHERE RUN_LABEL = '{run_label}'").collect()
    existing = ", ".join(r[0] for r in session.sql(f"SELECT DISTINCT TOPIC_ID FROM {DB}.ML.TOPIC_TAXONOMY ORDER BY 1").collect())
    session.sql(f"""
INSERT INTO {DB}.ML.PROPOSED_TOPICS
WITH ranked AS (
  SELECT c.CLUSTER_ID, e.REDACTED_CUSTOMER_TEXT,
         ROW_NUMBER() OVER (PARTITION BY c.CLUSTER_ID ORDER BY c.STRENGTH DESC, c.SEGMENT_ID) AS RN,
         COUNT(*) OVER (PARTITION BY c.CLUSTER_ID) AS MEMBERS
  FROM {DB}.ML.CLUSTER_MEMBERS c JOIN {DB}.CORE.SEGMENT_EMBEDDINGS e USING (SEGMENT_ID)
  WHERE c.RUN_LABEL = '{run_label}' AND c.CLUSTER_ID >= 0),
reps AS (
  SELECT CLUSTER_ID, ANY_VALUE(MEMBERS) AS MEMBERS,
         LISTAGG('- ' || LEFT(REDACTED_CUSTOMER_TEXT, 300), '\\n') WITHIN GROUP (ORDER BY RN) AS SAMPLES
  FROM ranked WHERE RN <= 6 GROUP BY 1),
named AS (
  SELECT CLUSTER_ID, MEMBERS,
         AI_COMPLETE(model => '{LLM}',
           prompt => 'Customer remarks from sales and success calls, grouped by similarity:\\n' || SAMPLES ||
                     '\\nExisting topics: {existing}. Name the shared theme in 2-5 words, describe it in one sentence, '
                     || 'and say which existing topic it duplicates, or NONE if it is new.',
           response_format => {{'type': 'json', 'schema': {{'type': 'object', 'properties': {{
             'label': {{'type': 'string'}}, 'description': {{'type': 'string'}},
             'duplicates': {{'type': 'string'}}}}, 'required': ['label', 'description', 'duplicates']}}}}) AS R
  FROM reps)
SELECT '{run_label}', CLUSTER_ID, MEMBERS, R:label::STRING, R:description::STRING, UPPER(R:duplicates::STRING),
       'PENDING', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, NULL
FROM named""").collect()
    return q(f"SELECT CLUSTER_ID, MEMBERS, LABEL, DUPLICATES, DESCRIPTION FROM {DB}.ML.PROPOSED_TOPICS "
             f"WHERE RUN_LABEL = '{run_label}' ORDER BY MEMBERS DESC")


PROPOSED_V1 = name_clusters("V1")
PROPOSED_V1

In [ ]:
n_clusters = int(V1["clusters"])
new_ones = PROPOSED_V1[PROPOSED_V1.DUPLICATES.str.contains("NONE")]
check("S5", len(PROPOSED_V1) == n_clusters and (PROPOSED_V1.LABEL.str.len() > 0).all() and len(new_ones) >= 1,
      f"{len(PROPOSED_V1)} proposals PENDING review ({n_clusters} AI_COMPLETE calls for {V1['segments']} leftover segments); "
      f"flagged new: {new_ones.LABEL.tolist()}")

---
## 6. January: run 2, matched to run 1

Discovery re-runs on all leftovers including January and logs **V2**. Cluster ids are arbitrary per run, so each V1
cluster is matched to the V2 cluster that shares most of its members (Jaccard overlap on the segments both runs saw), and
the mapping is kept in `ML.TOPIC_VERSION_MAP`. A reviewed name follows its topic across versions.

In [ ]:
%%sql -r discover_run_v2
CALL TRANSCRIPT_CHURN_ML_DEMO.ML.DISCOVER_TOPICS('V2', '2100-01-01')

In [ ]:
%%sql -r topic_version_map
CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_VERSION_MAP COMMENT = 'Each V1 cluster and its best-overlap V2 cluster' AS
WITH a AS (SELECT SEGMENT_ID, CLUSTER_ID FROM TRANSCRIPT_CHURN_ML_DEMO.ML.CLUSTER_MEMBERS WHERE RUN_LABEL = 'V1'),
b AS (SELECT SEGMENT_ID, CLUSTER_ID FROM TRANSCRIPT_CHURN_ML_DEMO.ML.CLUSTER_MEMBERS WHERE RUN_LABEL = 'V2'
      AND SEGMENT_ID IN (SELECT SEGMENT_ID FROM a)),
pairs AS (
  SELECT x.CLUSTER_ID AS V1_CLUSTER, y.CLUSTER_ID AS V2_CLUSTER,
         COUNT_IF(a.CLUSTER_ID = x.CLUSTER_ID AND b.CLUSTER_ID = y.CLUSTER_ID) /
         NULLIF(COUNT_IF(a.CLUSTER_ID = x.CLUSTER_ID OR b.CLUSTER_ID = y.CLUSTER_ID), 0) AS JACCARD
  FROM (SELECT DISTINCT CLUSTER_ID FROM a WHERE CLUSTER_ID >= 0) x
  CROSS JOIN (SELECT DISTINCT CLUSTER_ID FROM b WHERE CLUSTER_ID >= 0) y
  CROSS JOIN a JOIN b USING (SEGMENT_ID)
  GROUP BY 1, 2)
SELECT V1_CLUSTER, V2_CLUSTER, ROUND(JACCARD, 3) AS JACCARD
FROM pairs QUALIFY ROW_NUMBER() OVER (PARTITION BY V1_CLUSTER ORDER BY JACCARD DESC) = 1
ORDER BY V1_CLUSTER

In [ ]:
V2 = json.loads(discover_run_v2.iloc[0, 0])
PROPOSED_V2 = name_clusters("V2")
vm = q(f"SELECT * FROM {DB}.ML.TOPIC_VERSION_MAP")
session.sql(f"ALTER MODEL {DB}.ML.TOPIC_CLUSTERER SET DEFAULT_VERSION = V2").collect()
# The hidden topics are the ones that matter: their V1 clusters must survive into V2
hidden_v1 = q(f"""
WITH truth AS (SELECT DISTINCT CALL_ID || '-' || FLOOR((TURN_NO - 1) / 4) AS SEGMENT_ID, TRUE_TOPIC FROM {DB}.RAW.SEED_TRUTH)
SELECT c.CLUSTER_ID FROM {DB}.ML.CLUSTER_MEMBERS c JOIN truth t USING (SEGMENT_ID)
WHERE c.RUN_LABEL = 'V1' AND c.CLUSTER_ID >= 0 AND t.TRUE_TOPIC IN ('AI_LISTING_ASSISTANT', 'TEAM_SEAT_SHARING')
GROUP BY 1 HAVING COUNT(*) >= 10""").CLUSTER_ID.tolist()
STABILITY = vm[vm.V1_CLUSTER.isin(hidden_v1)].JACCARD.astype(float)
check("S6", len(STABILITY) >= 2 and STABILITY.min() >= 0.5 and V2["segments"] > V1["segments"],
      f"V2 on {V2['segments']} leftovers: {V2['clusters']} clusters, {V2['noise']} noise; emerging-topic clusters carried "
      f"V1->V2 with Jaccard {[round(x, 2) for x in STABILITY]} (>= 0.5); all pairs {vm.JACCARD.round(2).tolist()}; "
      f"default version V2")

---
## 7. One topic per segment

`ML.SEGMENT_TOPICS` is what every downstream consumer reads: the business topic when the seeded rule assigned one,
otherwise the discovered cluster from the registry's default version (`NEW_<cluster>` with its proposed label), otherwise
`UNASSIGNED`.

In [ ]:
%%sql -r segment_topics
CREATE OR REPLACE TABLE TRANSCRIPT_CHURN_ML_DEMO.ML.SEGMENT_TOPICS COMMENT = 'Final topic per segment: seeded, discovered (V2), or unassigned' AS
SELECT e.SEGMENT_ID, e.CALL_ID, e.ACCOUNT_ID, e.CALL_TS, e.CALL_TYPE, e.OPPORTUNITY_ID,
       CASE WHEN a.ASSIGNED THEN a.BEST_TOPIC
            WHEN c.CLUSTER_ID >= 0 THEN 'NEW_' || c.CLUSTER_ID ELSE 'UNASSIGNED' END AS TOPIC_ID,
       CASE WHEN a.ASSIGNED THEN t.TOPIC_NAME WHEN c.CLUSTER_ID >= 0 THEN p.LABEL ELSE 'Unassigned' END AS TOPIC_NAME,
       CASE WHEN a.ASSIGNED THEN 'SEEDED' WHEN c.CLUSTER_ID >= 0 THEN 'DISCOVERED' ELSE 'NOISE' END AS TOPIC_SOURCE,
       IFF(a.ASSIGNED, a.BEST_SIM, c.STRENGTH) AS CONFIDENCE
FROM TRANSCRIPT_CHURN_ML_DEMO.CORE.SEGMENT_EMBEDDINGS e
JOIN TRANSCRIPT_CHURN_ML_DEMO.ML.SEEDED_ASSIGNMENTS a USING (SEGMENT_ID)
LEFT JOIN (SELECT DISTINCT TOPIC_ID, TOPIC_NAME FROM TRANSCRIPT_CHURN_ML_DEMO.ML.TOPIC_TAXONOMY) t ON t.TOPIC_ID = a.BEST_TOPIC
LEFT JOIN TRANSCRIPT_CHURN_ML_DEMO.ML.CLUSTER_MEMBERS c ON c.SEGMENT_ID = e.SEGMENT_ID AND c.RUN_LABEL = 'V2'
LEFT JOIN TRANSCRIPT_CHURN_ML_DEMO.ML.PROPOSED_TOPICS p ON p.RUN_LABEL = 'V2' AND p.CLUSTER_ID = c.CLUSTER_ID

In [ ]:
%%sql -r topic_mix_by_month
SELECT TO_CHAR(DATE_TRUNC(month, CALL_TS), 'YYYY-MM') AS MONTH, TOPIC_SOURCE, TOPIC_NAME, COUNT(*) AS SEGMENTS
FROM TRANSCRIPT_CHURN_ML_DEMO.ML.SEGMENT_TOPICS WHERE TOPIC_SOURCE <> 'SEEDED'
GROUP BY 1, 2, 3 ORDER BY 1, 4 DESC

In [ ]:
st = q(f"SELECT TOPIC_SOURCE, COUNT(*) AS N FROM {DB}.ML.SEGMENT_TOPICS GROUP BY 1").set_index("TOPIC_SOURCE").N
total = one(f"SELECT COUNT(*) FROM {DB}.CORE.SEGMENT_EMBEDDINGS")
check("S7", int(st.sum()) == total and int(st.get("DISCOVERED", 0)) > 0 and int(st.get("NOISE", 0)) > 0,
      f"ML.SEGMENT_TOPICS {int(st.sum()):,} rows = embeddings {total:,}: seeded {int(st.get('SEEDED', 0)):,}, "
      f"discovered {int(st.get('DISCOVERED', 0)):,}, unassigned {int(st.get('NOISE', 0)):,}")

---
## Summary

Business topics are assigned in SQL; new ones are discovered with density clustering in a stored procedure, versioned in
the Model Registry, named by an LLM from a handful of representatives and held for review. Next: `03` turns topics into
the business answers (price negativity by driver, topics on won and lost deals).